### Gradient descent on the XOR problem, with and without bias

XOR is the classic first test of a hidden layer. No single straight line
separates the two rows that should output 1 from the two that should output 0,
so a network with no hidden layer cannot solve it at all.

This notebook trains the same 2-4-1 network twice. The first version gives each
neuron only a weighted sum of its inputs, $w \cdot x$. The second adds a **bias**
to every neuron, $w \cdot x + b$, so the point at which a neuron turns on can move
away from the origin.

Both versions are trained under two different weight initialization strategies,
**random normal** and **Xavier**, and both are swept across 100 learning rates so
the comparison is a curve rather than a single number.

In [ ]:
"""gradient_descent.ipynb"""

# Cell 01 - Import packages and define the network topology

%matplotlib inline

from enum import Enum, auto

import matplotlib.pyplot as plt
import numpy as np
import rich
from matplotlib.ticker import MultipleLocator
from rich import box
from rich.table import Table
from rich.theme import Theme

# Every table in this notebook prints in one plain color, with no cell
# emphasized over another. Nothing in how a number looks should suggest
# what it means: the numbers are compared against each other, and the
# markdown says what the comparison shows.
PLAIN = Theme(
    {
        "table.title": "none",
        "table.header": "none",
        "table.footer": "none",
        "table.caption": "none",
    }
)

# Reconfigure Rich's own console rather than building a separate one, so
# every table renders the same way. highlight=False stops Rich from
# coloring numbers on its own
rich.reconfigure(highlight=False, theme=PLAIN)
console = rich.get_console()


class InitStrategy(Enum):
    RND_NORMAL = auto()
    XAVIER = auto()


# Network topology: 2 inputs, one hidden layer of 4 neurons, 1 output
INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE = 2, 4, 1

# Both halves of the notebook train for the same number of epochs, so the
# only thing that differs between them is whether the neurons carry a bias
EPOCHS = 1_000

print(f"Topology:   {INPUT_SIZE}-{HIDDEN_SIZE}-{OUTPUT_SIZE}")
print(f"Epochs:     {EPOCHS:,}")
print(f"Strategies: {[s.name for s in InitStrategy]}")

---
### The XOR truth table

XOR outputs 1 when exactly one of its two inputs is 1. All four rows are the
entire problem, so there is no fifth case held back to test against. Whatever
the network learns here it learns about all of XOR.

In [ ]:
# Cell 02 - The four rows of the XOR truth table

x = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

table = Table(title="XOR truth table", box=box.SIMPLE, title_justify="left")
table.add_column("x1", justify="right")
table.add_column("x2", justify="right")
table.add_column("y", justify="right")

for inputs, target in zip(x, y):
    table.add_row(str(inputs[0]), str(inputs[1]), str(target[0]))

console.print(table)

---
### The sigmoid activation function

Every neuron in this network passes its weighted sum through the logistic sigmoid

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

which squashes any real number into the open interval $(0, 1)$.

Backpropagation needs its derivative. Written in terms of the sigmoid's own
output $s = \sigma(z)$, that derivative is simply $s(1 - s)$, so there is no need
to keep the original input around. Notice the derivative peaks at $z = 0$ and
falls toward zero at both extremes: a neuron driven far from zero learns slowly.

In [ ]:
# Cell 03 - Sigmoid activation and its derivative


def sigmoid(z: np.ndarray) -> np.ndarray:
    """Squash every value into the interval (0, 1)."""
    return 1 / (1 + np.exp(-z))


def sigmoid_derivative(s: np.ndarray) -> np.ndarray:
    """Derivative of the sigmoid, given s = sigmoid(z) rather than z."""
    return s * (1 - s)


# Both functions are vectorized, so one call handles the whole sample
z = np.array([-4, -1, 0, 1, 4])
s = sigmoid(z)
d = sigmoid_derivative(s)

table = Table(title="Sigmoid and its derivative", box=box.SIMPLE, title_justify="left")
table.add_column("z", justify="right")
table.add_column("sigmoid(z)", justify="right")
table.add_column("derivative", justify="right")

for value, activation, slope in zip(z, s, d):
    table.add_row(str(value), f"{activation:.4f}", f"{slope:.4f}")

console.print(table)

---
### Initializing the weights and biases

Two strategies are compared throughout the notebook:

- **Random normal** draws every weight from the standard normal distribution,
  ignoring how many connections feed into each neuron.
- **Xavier** (Glorot and Bengio, 2010) draws uniformly from
  $[-\text{limit}, +\text{limit}]$ where $\text{limit} = \sqrt{6 / (n_{in} + n_{out})}$.
  Scaling by the layer sizes keeps the signal from growing or shrinking as it
  passes through the network.

The seed is set *before* the match rather than inside one branch, so both
strategies start from the same point in the random stream. That makes the two
curves plotted later a fair comparison: the only thing that differs between two
runs is what we deliberately changed.

Biases always start at zero. Unlike the weights they do not need random values
to break symmetry, because the incoming weights already differ neuron to neuron.

In [ ]:
# Cell 04 - Initialize the weight matrices and bias vectors


def init_weights(init_strategy: InitStrategy) -> tuple[np.ndarray, ...]:
    """Return W_ih, W_ho, b_ih, b_ho for the chosen strategy.

    Both training functions call this, so the two halves of the notebook
    start from identical weights. The version without bias simply ignores
    the two bias vectors it is handed.
    """
    # Seed before the match so every strategy starts from the same point
    np.random.seed(2020)

    match init_strategy:
        case InitStrategy.RND_NORMAL:
            W_ih = np.random.randn(INPUT_SIZE, HIDDEN_SIZE)
            W_ho = np.random.randn(HIDDEN_SIZE, OUTPUT_SIZE)

        case InitStrategy.XAVIER:
            limit = np.sqrt(6 / (INPUT_SIZE + HIDDEN_SIZE))
            W_ih = np.random.uniform(-limit, limit, size=(INPUT_SIZE, HIDDEN_SIZE))
            limit = np.sqrt(6 / (HIDDEN_SIZE + OUTPUT_SIZE))
            W_ho = np.random.uniform(-limit, limit, size=(HIDDEN_SIZE, OUTPUT_SIZE))

        case _:
            # Every strategy must set both weight matrices, so an unhandled
            # one is a mistake worth reporting here rather than later
            raise ValueError(f"Unknown init strategy: {init_strategy}")

    b_ih = np.zeros((1, HIDDEN_SIZE))
    b_ho = np.zeros((1, OUTPUT_SIZE))

    return W_ih, W_ho, b_ih, b_ho


table = Table(title="Starting weights", box=box.SIMPLE, title_justify="left")
table.add_column("strategy")
table.add_column("W_ih shape", justify="right")
table.add_column("smallest", justify="right")
table.add_column("largest", justify="right")
table.add_column("biases", justify="right")

for strategy in InitStrategy:
    W_ih, W_ho, b_ih, b_ho = init_weights(strategy)
    table.add_row(
        strategy.name,
        str(W_ih.shape),
        f"{W_ih.min():+.3f}",
        f"{W_ih.max():+.3f}",
        "all zero" if not b_ih.any() else "nonzero",
    )

console.print(table)

---
### Training without bias

One epoch is one pass over all four rows at once:

1. **Forward.** Each layer computes a dot product and applies the sigmoid.
2. **Residual.** How far each output sits from its target, $y - \hat{y}$.
3. **Backward.** Convert the residual into a *delta* per layer by multiplying
   through the sigmoid derivative, then step every weight in the direction that
   reduces the loss.

The loss reported is $\frac{1}{2}(y - \hat{y})^2$ averaged over the four rows.

In [ ]:
# Cell 05 - Train the network with no bias terms


def train(
    init_strategy: InitStrategy, learning_rate: float = 1.0
) -> tuple[float, np.ndarray]:
    """Train the 2-4-1 network without bias, returning its loss and outputs."""
    W_ih, W_ho, _, _ = init_weights(init_strategy)

    # Give the loss a starting value so it is defined even if EPOCHS is 0
    loss = np.zeros_like(y, dtype=float)

    for _ in range(EPOCHS):
        # Forward pass
        hidden_output = sigmoid(np.dot(x, W_ih))
        final_output = sigmoid(np.dot(hidden_output, W_ho))

        # Compute the loss function
        residual = y - final_output
        loss = 0.5 * residual**2

        # Backpropagate the loss and update the weights
        output_delta = residual * sigmoid_derivative(final_output)
        W_ho += learning_rate * np.dot(hidden_output.T, output_delta)

        hidden_error = np.dot(output_delta, W_ho.T)
        hidden_delta = hidden_error * sigmoid_derivative(hidden_output)
        W_ih += learning_rate * np.dot(x.T, hidden_delta)

    final_output = sigmoid(np.dot(sigmoid(np.dot(x, W_ih)), W_ho))

    # np.mean returns np.float64, so convert it to match the declared type
    return float(np.mean(loss)), final_output


# Quick check that the function runs and learns something
check_loss, _ = train(InitStrategy.XAVIER)
print(f"train(XAVIER) final loss = {check_loss:.5f}")

---
### What the network without bias actually answers

Read the four columns against the target row underneath. A value near 0.5 means
the network never committed to an answer for that input.

In [ ]:
# Cell 06 - Run both initialization strategies without bias


def run_model(train_fn, learning_rate: float = 1.0) -> None:
    """Train once per strategy and show the four outputs side by side."""
    table = Table(
        title=f"Network output at learning rate {learning_rate}",
        box=box.SIMPLE_HEAVY,
        title_justify="left",
    )
    table.add_column("strategy")
    for row in x:
        table.add_column(f"[{row[0]},{row[1]}]", justify="right")
    table.add_column("loss", justify="right")

    for strategy in InitStrategy:
        final_loss, final_output = train_fn(strategy, learning_rate)
        table.add_row(
            strategy.name,
            *[f"{value:.3f}" for value in final_output.ravel()],
            f"{final_loss:.5f}",
        )

    # The targets go below a divider, as the answer the rows are aiming at
    table.add_section()
    table.add_row("target", *[str(value) for value in y.ravel()], "")

    console.print(table)


run_model(train)

---
### Learning rate versus final loss, without bias

A single learning rate proves nothing, so the next cell retrains the network
from scratch at 100 different learning rates between 0.1 and 1.0. Every point on
the curve is an independent training run: a learning rate of 0.2 is not a
continuation of 0.1, it is a separate experiment.

Lower is better on the vertical axis.

In [ ]:
# Cell 07 - Sweep the learning rate and plot the final loss, without bias


def plot_models(train_fn, title: str) -> tuple[np.ndarray, np.ndarray]:
    """Retrain at 100 learning rates and plot the final loss of each run."""
    learning_rate = np.linspace(0.1, 1.0, 100)
    loss_rnd = np.zeros_like(learning_rate)
    loss_xavier = np.zeros_like(learning_rate)

    # Every point is a network trained from scratch at that learning rate
    for i, rate in enumerate(learning_rate):
        loss_rnd[i] = train_fn(InitStrategy.RND_NORMAL, rate)[0]
        loss_xavier[i] = train_fn(InitStrategy.XAVIER, rate)[0]

    plt.figure(figsize=(8, 5))
    plt.plot(learning_rate, loss_rnd, label="RND Normal")
    plt.plot(learning_rate, loss_xavier, label="Xavier")
    plt.title(f"{title} ({EPOCHS:,} Epochs)\nLearning Rate vs. Final Loss")
    plt.xlabel("Learning Rate")
    plt.ylabel("Final Loss")
    plt.gca().xaxis.set_major_locator(MultipleLocator(0.1))
    plt.legend()
    plt.grid()
    plt.show()

    return loss_rnd, loss_xavier


rnd_no_bias, xavier_no_bias = plot_models(train, "XOR NN (2-4-1) w/o Bias")
print(
    f"median loss   RND Normal {np.median(rnd_no_bias):.5f}"
    f"    Xavier {np.median(xavier_no_bias):.5f}"
)

---
### Adding a bias to every neuron

Without a bias a neuron fires on $w \cdot x > 0$, so its dividing line must pass
through the origin. Adding $b$ gives $w \cdot x + b > 0$, which lets the same
neuron slide that line anywhere it needs to be.

The `[0, 0]` row shows why this matters. That input contributes nothing to any
dot product, so without a bias the hidden layer receives all zeros no matter what
the input weights are, and its response is pinned regardless of training.

The training loop below is identical apart from two changes: the forward pass
adds `b_ih` and `b_ho`, and the backward pass updates them. A weight update is
scaled by its incoming activation, but a bias has no input to scale it, so its
gradient is just the delta summed over the rows of the batch.

In [ ]:
# Cell 08 - Train the network with a bias on every neuron


def train_with_bias(
    init_strategy: InitStrategy, learning_rate: float = 1.0
) -> tuple[float, np.ndarray]:
    """Train the 2-4-1 network with bias, returning its loss and outputs."""
    W_ih, W_ho, b_ih, b_ho = init_weights(init_strategy)

    loss = np.zeros_like(y, dtype=float)

    for _ in range(EPOCHS):
        # Forward pass, now adding a bias before each activation
        hidden_output = sigmoid(np.dot(x, W_ih) + b_ih)
        final_output = sigmoid(np.dot(hidden_output, W_ho) + b_ho)

        # Compute the loss function
        residual = y - final_output
        loss = 0.5 * residual**2

        # Backpropagate the loss and update the weights
        output_delta = residual * sigmoid_derivative(final_output)
        W_ho += learning_rate * np.dot(hidden_output.T, output_delta)

        hidden_error = np.dot(output_delta, W_ho.T)
        hidden_delta = hidden_error * sigmoid_derivative(hidden_output)
        W_ih += learning_rate * np.dot(x.T, hidden_delta)

        # Update the biases: no incoming activation to scale the gradient
        b_ih += learning_rate * np.sum(hidden_delta, axis=0, keepdims=True)
        b_ho += learning_rate * np.sum(output_delta, axis=0, keepdims=True)

    hidden_output = sigmoid(np.dot(x, W_ih) + b_ih)
    final_output = sigmoid(np.dot(hidden_output, W_ho) + b_ho)

    # np.mean returns np.float64, so convert it to match the declared type
    return float(np.mean(loss)), final_output


# Quick check, to compare against the same call without bias above
check_loss, _ = train_with_bias(InitStrategy.XAVIER)
print(f"train_with_bias(XAVIER) final loss = {check_loss:.5f}")

---
### What the network with bias answers

Same table as before, produced by the same `run_model` helper. Compare the
`[1,1]` column in particular against the run without bias.

In [ ]:
# Cell 09 - Run both initialization strategies with bias

run_model(train_with_bias)

---
### Learning rate versus final loss, with bias

The same 100-point sweep, the same helper, the same epoch budget. Only the
training function changed. Note the vertical scale before comparing this plot
with the previous one.

In [ ]:
# Cell 10 - Sweep the learning rate and plot the final loss, with bias

rnd_bias, xavier_bias = plot_models(train_with_bias, "XOR NN (2-4-1) with Bias")
print(
    f"median loss   RND Normal {np.median(rnd_bias):.5f}"
    f"    Xavier {np.median(xavier_bias):.5f}"
)

---
### Reading the two experiments together

The final cell puts all four curves side by side. Two things to draw out:

- **Bias lowers the loss by roughly a factor of 10** at the same epoch budget,
  and it does so at every learning rate rather than at a lucky few.
- **Bias also makes the initialization strategy matter less.** Without bias the
  gap between random normal and Xavier is large, so a good starting point has to
  rescue the network. With bias the two strategies land close together.

XOR is still solvable without bias given enough epochs, so what the bias really
buys here is speed and reliability, not possibility.

In [ ]:
# Cell 11 - Compare all four sweeps in one table

results = {
    ("w/o bias", "RND Normal"): rnd_no_bias,
    ("w/o bias", "Xavier"): xavier_no_bias,
    ("with bias", "RND Normal"): rnd_bias,
    ("with bias", "Xavier"): xavier_bias,
}

table = Table(
    title="Final loss across 100 learning rates",
    box=box.SIMPLE_HEAVY,
    title_justify="left",
)
table.add_column("network")
table.add_column("strategy")
table.add_column("best", justify="right")
table.add_column("median", justify="right")
table.add_column("worst", justify="right")

for (network, strategy), losses in results.items():
    table.add_row(
        network,
        strategy,
        f"{losses.min():.5f}",
        f"{np.median(losses):.5f}",
        f"{losses.max():.5f}",
    )

console.print(table)

improvement = np.median(rnd_no_bias) / np.median(rnd_bias)
console.print(f"Adding bias lowered the median loss by a factor of {improvement:.1f}")